In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.signal import find_peaks, welch
import pandas as pd
from pandas.plotting import parallel_coordinates
import sklearn
from sklearn.cluster import KMeans
from lib.readwav import *

# Estrapolazione e salvataggio delle armoniche nelle note prodotte dall'organo


In [2]:
# Funzione per la stima più precisa della fondamentale:

def f0_MCD(frequencies_of_peaks, highest_candidate):
    lowest_candidate = np.min(frequencies_of_peaks) / 3
    candidates = np.arange(highest_candidate, lowest_candidate-0.1, -0.1)
    lowest_residual = float("inf")
    best_f0 = highest_candidate

    for f0 in candidates:
        ratios = frequencies_of_peaks / f0
        integers = np.round(ratios)
        if np.any(integers == 0):
            continue
        discrepancies = np.abs(ratios - integers)
        residual = np.sum(discrepancies)
        found_harmonics = len(np.unique(integers))
        max_harmonic = np.max(integers)
        missed_harmonics = max_harmonic - found_harmonics
        residual += missed_harmonics * 0.1
        if residual < (lowest_residual * 0.95):
            lowest_residual = residual
            best_f0 = f0
    
    return best_f0

In [3]:
# Ciclo per estrapolazione su tutti i file:

fundamentals = [] # Array contenente le sole fondamentali per ciascun file
harmonics = [] # Lista contenente le armoniche: ogni riga è un file, ogni colonna un'armonica (inclusa la fondamentale)
power_spectra = [] # Lista contenente gli spettri valutati nelle frequenze delle armoniche
normalized_power_spectra = [] # Lista contenente sempre gli spettri valutati, ma ora anche divisi per la potenza della fondamentale

for file in range(280):
    print("È in corso l'analisi del file",file+1,"...")
    
    # Lettura file:
    filename = "./wav_files/note_{}.wav".format(file+1)
    rate, note = readwav(filename)
    note = note[:,0]
    time = np.linspace(0, note.shape[0], note.shape[0], endpoint = False) / rate
    first_15 = time[0] + 0.15 * (time[-1] - time[0])
    last_5 = time[0] + 0.95 * (time[-1] - time[0])
    cut = (time >= first_15) & (time <= last_5)
    time = time[cut]
    note = note[cut]
   
    # Calcolo preventivo della fondamentale:
    width = (time[-1] - time[0]) / 10
    gaussian_window = scipy.signal.windows.gaussian(int(width * rate), std = int(width * rate / 6))
    frequencies, spectrum = welch(note, fs = rate, window = gaussian_window, nperseg = len(gaussian_window))
    spectrum = 10 * np.log10(spectrum)
    peaks, _ = find_peaks(spectrum, prominence = 20)
    index_of_max = np.argmax(spectrum[peaks])
    previsional_frequency = frequencies[peaks[index_of_max]]    

    # Calcolo più preciso della fondamentale:
    frequencies_of_peaks = frequencies[peaks]
    frequencies_of_peaks = frequencies_of_peaks[:10] # La prominence è bassina, quindi ad alte frequenza prende un sacco di rumore
    fundamental = f0_MCD(frequencies_of_peaks, previsional_frequency)

    # Calcolo delle 5 armoniche successive:
    estimated_harmonics = [fundamental * i for i in range(1,7)] # 6 elementi perché considereremo le prime 6 armoniche
    measured_harmonics_frequency = []
    measured_harmonics_power = []
    neighborhood = 2
    for estimate in estimated_harmonics:
        index_closest_harmonic = np.argmin(np.abs(estimate - frequencies_of_peaks))
        closest_harmonic = frequencies_of_peaks[index_closest_harmonic]
        if abs(estimate - closest_harmonic) < (0.05 * closest_harmonic):
            measured_harmonics_frequency.append(closest_harmonic)
            index_target = np.argmin(np.abs(closest_harmonic - frequencies))
            measured_harmonics_power.append(spectrum[index_target])
        else:
            measured_harmonics_frequency.append(estimate)
            index_target = np.argmin(np.abs(estimate - frequencies))
            start = max(0, index_target - neighborhood)
            end = min(len(spectrum), index_target + neighborhood + 1)
            measured_harmonics_power.append(np.max(spectrum[start:end]))

    # Aggiornamento degli array di scope globale:
    fundamentals.append(fundamental)
    harmonics.append(measured_harmonics_frequency)
    power_spectra.append(measured_harmonics_power)
    normalized_power_spectra.append(measured_harmonics_power - measured_harmonics_power[0])

print("\n\nFinito!\n")

È in corso l'analisi del file 1 ...
È in corso l'analisi del file 2 ...
È in corso l'analisi del file 3 ...
È in corso l'analisi del file 4 ...
È in corso l'analisi del file 5 ...
È in corso l'analisi del file 6 ...
È in corso l'analisi del file 7 ...
È in corso l'analisi del file 8 ...
È in corso l'analisi del file 9 ...
È in corso l'analisi del file 10 ...
È in corso l'analisi del file 11 ...
È in corso l'analisi del file 12 ...
È in corso l'analisi del file 13 ...
È in corso l'analisi del file 14 ...
È in corso l'analisi del file 15 ...
È in corso l'analisi del file 16 ...
È in corso l'analisi del file 17 ...
È in corso l'analisi del file 18 ...
È in corso l'analisi del file 19 ...
È in corso l'analisi del file 20 ...
È in corso l'analisi del file 21 ...
È in corso l'analisi del file 22 ...
È in corso l'analisi del file 23 ...
È in corso l'analisi del file 24 ...
È in corso l'analisi del file 25 ...
È in corso l'analisi del file 26 ...
È in corso l'analisi del file 27 ...
È in corso

In [4]:
# Salvataggio su file esterni:
np.savetxt("fundamentals.csv", fundamentals, delimiter = ",")
np.savetxt("harmonics.csv", harmonics, delimiter = ",")
np.savetxt("power_spectra.csv", power_spectra, delimiter = ",")
np.savetxt("normalized_power_spectra.csv", normalized_power_spectra, delimiter = ",")